# 3 · Other catalogs and reusable exports

Earth Search and Planetary Computer are included as public STAC API presets. A custom root must use HTTPS, return a STAC Catalog or Collection, and allow browser CORS. Public metadata access does not guarantee unrestricted asset access; some Planetary Computer assets require separate signing. This notebook does not fetch raster assets.

In [ ]:
from stac_browser import STACBrowser, PRESETS
client = await STACBrowser(PRESETS['earthsearch']).connect()
print('Catalog:', client.root.get('title', client.root['id']))
print('Collections:', [c['id'] for c in client.collections])

In [ ]:
items = await client.search(bbox=[-123.1, 37.2, -121.8, 38.1], collection='sentinel-2-l2a', start='2024-06-01', end='2024-06-15', limit=3)
print(len(items), 'items loaded')
if items:
    client.save('earthsearch')
    print('Saved earthsearch-items.geojson and earthsearch-search.json')

## Reuse exported search JSON

The map and Python helper export `stac_root`, `query`, `requests`, retrieval time, and count. Upload the downloaded JSON to JupyterLite. Replace `saved` below with `json.loads(Path('stac-search.json').read_text(encoding='utf-8'))` to reuse an exported query. Dates come from the inclusive day interval exported by this project.

Static-item exports have `query: null`: their provenance records the item GET request instead of an API search. The replay helper handles that branch first and returns the loaded item as a one-element list. API exports repeat the first search page; call `more()` on a connected client to request further pages. GeoJSON export copies resolve relative links to absolute URLs while the client retains the original metadata.

In [ ]:
import json
from pathlib import Path
saved = client.provenance()
print('Source root:', saved['stac_root'])
print('Reproducible query:', saved.get('query'))

async def replay_saved(saved):
    replay = await STACBrowser(saved['stac_root']).connect()
    query = saved.get('query')
    if query is None:
        requests = saved.get('requests') or []
        if not requests or requests[-1].get('method', 'GET').upper() != 'GET':
            raise ValueError('Static-item replay requires an exported item GET request.')
        await replay.open_link({'rel': 'item', 'href': requests[-1]['url']})
        return replay.items
    dates = query.get('datetime', '../..').split('/')
    return await replay.search(
        bbox=query['bbox'], collection=(query.get('collections') or [None])[0],
        start=None if dates[0] == '..' else dates[0][:10],
        end=None if dates[1] == '..' else dates[1][:10], limit=query['limit'])

# To issue the saved request again (new HTTP requests), uncomment:
# replay_items = await replay_saved(saved)

## Static catalogs are different

A static Catalog/Collection may have `child` and `item` links without a remote `search` endpoint. Connect with `await STACBrowser('https://.../catalog.json').connect()`, inspect `client.browse_links()`, and pass a selected link to `await client.open_link(link)`. Child links return another connected browser; item links load one item for export. This does not pretend to provide server-side search or recursively crawl the catalog.

## Troubleshooting

A request failure can mean CORS, offline networking, an invalid TLS certificate, a provider error, or non-JSON content. There is no CORS bypass. Correct the URL, choose another preset, or retry later. Dates, bbox order, and page sizes are validated before requests. The map’s **Field guide** link includes architecture and references.